In [1]:
import re
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

In [2]:
def tokenize(text):
    return re.findall(r"[^\W\d_]+", text.lower(), flags=re.UNICODE)

def mapper(text):
    return [(word, 1) for word in tokenize(text)]

def combiner(pairs):
    counts = {}
    for word, count in pairs:
        counts[word] = counts.get(word, 0) + count
    return counts

def partitioner(word, num_reducers=2):
    return sum(ord(c) for c in word) % num_reducers

def reducer(values):
    return sum(values)

In [3]:
def run_mapreduce(files, num_reducers=2):
    def map_file(path):
        text = Path(path).read_text(encoding="utf-8")
        return combiner(mapper(text))

    with ThreadPoolExecutor() as executor:
        local_results = list(executor.map(map_file, files))

    shuffled = [defaultdict(list) for _ in range(num_reducers)]
    for counts in local_results:
        for word, count in counts.items():
            r_id = partitioner(word, num_reducers)
            shuffled[r_id][word].append(count)

    reduced = []
    for r_dict in shuffled:
        res = {w: reducer(counts) for w, counts in sorted(r_dict.items())}
        reduced.append(res)

    return reduced

In [ ]:
files = sorted(Path("../datasets/text-files").glob("*.txt"))
print(len(files), "files found")   # should not be 0
results = run_mapreduce(files)

for i, res in enumerate(results):
    print(f"Reducer {i} output ({len(res)} unique words):")
    for word, count in list(res.items())[:10]:
        print(f"  {word}: {count}")

Reducer 0 output (0 unique words):
Reducer 1 output (0 unique words):
